# VisionGuard AI — Phase 3.3: RDD2022 Dataset Preparation & Validation for YOLOv8

**Project:** VisionGuard AI — Automated Infrastructure Crack & Pothole Inspection  
**Phase:** Phase 3.3 — Google Colab Dataset Acquisition, Conversion, and Validation  
**Target Architecture:** YOLOv8 Nano (`yolov8n`)  
**Dataset:** Road Damage Dataset 2022 (RDD2022 / CRDDC'2022)  
**Target Distress Classes:** `D00` (Longitudinal Crack), `D10` (Transverse Crack), `D20` (Alligator Crack), `D40` (Pothole)  
**Execution Scope:** Dataset acquisition, validation, sequence splitting, and configuration ONLY. **No model training occurs in this notebook.**

---

### Instructions for Running in Google Colab
1. **Select GPU Runtime:** Go to **Runtime** > **Change runtime type**, select **T4 GPU** under Hardware accelerator, and click **Save**.
2. **Execute Sequentially:** Run cells from top to bottom. Do not skip verification cells.
3. **Ephemeral Storage Notice:** `/content` local storage is temporary and will reset when the session disconnects. In Step 14, you can optionally back up the final prepared dataset directly to your personal Google Drive.

## Step 1: Runtime and Hardware Verification

Check that the active Google Colab environment has allocated an NVIDIA GPU (typically a Tesla T4 with ~15 GB VRAM) and that PyTorch has CUDA access.

In [ ]:
# 1. Check GPU hardware and VRAM allocation
import os
import subprocess
import torch

print("=== Google Colab GPU & System Diagnostics ===")
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"Active GPU Device:      {gpu_name}")
    print(f"Total Dedicated VRAM:   {total_vram_gb} GB")
    if total_vram_gb < 10.0:
        print("WARNING: VRAM is less than 10 GB. Consider switching to T4 GPU runtime.")
else:
    print("\n[CRITICAL WARNING] No GPU detected! Please configure Colab:")
    print("  -> Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

## Step 2: Package Installation and Dependency Verification

Verify the versions of core computer vision packages (`ultralytics`, `opencv`, `pillow`, `pyyaml`, `tqdm`, `requests`).

In [ ]:
# 2. Install and verify required packages in Colab environment
%pip install -q --upgrade ultralytics pillow opencv-python matplotlib pyyaml tqdm requests

import sys
import PIL
import cv2
import matplotlib
import yaml
import requests
import ultralytics

print("=== Dependency Verification ===")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Ultralytics:  {ultralytics.__version__}")
print(f"OpenCV:       {cv2.__version__}")
print(f"Pillow (PIL): {PIL.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")
print(f"PyYAML:       {yaml.__version__}")
print("All core dependencies loaded successfully.")

## Step 3: Dataset Provenance, Licensing & API Verification

Query the official Figshare API endpoint (`article ID: 21431547`) to confirm live availability, checksums, and license compliance before initiating downloads.

In [ ]:
# 3. Verify dataset metadata and live access via Figshare API
import requests
import json

ARTICLE_ID = 21431547
FIGSHARE_API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"

print(f"Querying official Figshare repository endpoint: {FIGSHARE_API_URL} ...")
response = requests.get(FIGSHARE_API_URL, timeout=30)
assert response.status_code == 200, f"Failed to access Figshare API! HTTP Status: {response.status_code}"

metadata = response.json()
print("\n=== Dataset Provenance & Attribution ===")
print(f"Title:        {metadata.get('title')}")
print(f"DOI:          {metadata.get('doi')}")
print(f"License:      {metadata.get('license', {}).get('name')} ({metadata.get('license', {}).get('url')})")
print(f"Citation:     {metadata.get('citation')}")

files = metadata.get('files', [])
print(f"\nRepository Files ({len(files)} items):")
target_archive_info = None
for f in files:
    print(f" - {f['name']} ({round(f['size'] / 1e9, 2)} GB) [ID: {f['id']}]")
    if f['name'] == 'RDD2022_released_through_CRDDC2022.zip':
        target_archive_info = f

assert target_archive_info is not None, "Target archive RDD2022_released_through_CRDDC2022.zip not found!"
print("\n[PASSED] Official archive verified on Figshare.")
print(f"  Download URL: {target_archive_info['download_url']}")
print(f"  Expected MD5: {target_archive_info['computed_md5']}")

## Step 4: Ephemeral Storage Assessment & Dataset Download

Check available ephemeral NVMe storage in `/content` and download `RDD2022_released_through_CRDDC2022.zip` (~13.26 GB) with chunked streaming.

In [ ]:
# 4. Check available storage and stream download
import shutil
import os
from tqdm import tqdm

total_b, used_b, free_b = shutil.disk_usage('/content')
free_gb = free_b / (1024**3)
print(f"Available ephemeral disk space in /content: {free_gb:.2f} GB")
assert free_gb >= 25.0, f"Insufficient disk space! Needed >= 25 GB, found {free_gb:.2f} GB."

ARCHIVE_PATH = '/content/RDD2022_released_through_CRDDC2022.zip'
DOWNLOAD_URL = target_archive_info['download_url']
EXPECTED_BYTES = target_archive_info['size']

if not os.path.exists(ARCHIVE_PATH) or os.path.getsize(ARCHIVE_PATH) != EXPECTED_BYTES:
    print(f"\nDownloading {round(EXPECTED_BYTES / 1e9, 2)} GB archive to {ARCHIVE_PATH}...")
    with requests.get(DOWNLOAD_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', EXPECTED_BYTES))
        with open(ARCHIVE_PATH, 'wb') as f, tqdm(
            total=total_size, unit='B', unit_scale=True, desc='RDD2022.zip'
        ) as pbar:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    print("Download completed.")
else:
    print(f"\nArchive already exists at {ARCHIVE_PATH} ({round(os.path.getsize(ARCHIVE_PATH) / 1e9, 2)} GB).")

## Step 5: Archive Integrity Verification

Calculate the MD5 checksum of the downloaded file and run a zip integrity test to guarantee that no bytes were corrupted during transit.

In [ ]:
# 5. Checksum and zip structure verification
import hashlib
import zipfile

print("Calculating MD5 checksum (this may take ~1 minute for 13 GB)...")
md5_hash = hashlib.md5()
with open(ARCHIVE_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        md5_hash.update(chunk)

computed_md5 = md5_hash.hexdigest()
expected_md5 = target_archive_info['computed_md5']

print(f"Computed MD5: {computed_md5}")
print(f"Expected MD5: {expected_md5}")
assert computed_md5 == expected_md5, "MD5 checksum mismatch! Download is incomplete or corrupt."
print("[PASSED] MD5 checksum matched perfectly.")

print("Testing zip structure...")
with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
    bad_file = zf.testzip()
    assert bad_file is None, f"Corrupted entry inside archive: {bad_file}"
print("[PASSED] Zip archive integrity verified.")

## Step 6: Selective Extraction of Chosen Subset (Japan Train)

Rather than extracting all ~25 GB of uncompressed data across 6 countries and unannotated benchmarks, selectively extract only the **Japan Train Subset** (`RDD2022/Japan/train/`).

In [ ]:
# 6. Selective extraction of RDD2022/Japan/train/
EXTRACT_ROOT = '/content/raw_rdd'
TARGET_PREFIX = 'RDD2022/Japan/train/'

os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
    namelist = zf.namelist()
    japan_train_members = [
        name for name in namelist 
        if name.startswith(TARGET_PREFIX) and not name.endswith('/')
    ]
    print(f"Total items in archive: {len(namelist):,}")
    print(f"Target subset items to extract: {len(japan_train_members):,}")
    
    print("\nExtracting Japan train subset...")
    for member in tqdm(japan_train_members, desc="Extracting Japan train"):
        zf.extract(member, path=EXTRACT_ROOT)

RAW_IMG_DIR = os.path.join(EXTRACT_ROOT, 'RDD2022/Japan/train/images')
RAW_XML_DIR = os.path.join(EXTRACT_ROOT, 'RDD2022/Japan/train/annotations/xmls')

print(f"\nExtracted Images:      {RAW_IMG_DIR} ({len(os.listdir(RAW_IMG_DIR)):,} files)")
print(f"Extracted Annotations: {RAW_XML_DIR} ({len(os.listdir(RAW_XML_DIR)):,} files)")

## Step 7: Image and Annotation Pairing Audit

Verify that each JPEG image has a matching Pascal VOC XML file and test that image files open cleanly.

In [ ]:
# 7. Verify 1:1 image and annotation pairing
from PIL import Image

img_files = set(f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg')))
xml_files = set(f for f in os.listdir(RAW_XML_DIR) if f.lower().endswith('.xml'))

img_stems = set(os.path.splitext(f)[0] for f in img_files)
xml_stems = set(os.path.splitext(f)[0] for f in xml_files)

common_stems = sorted(list(img_stems & xml_stems))
unmatched_imgs = img_stems - xml_stems
unmatched_xmls = xml_stems - img_stems

print("=== Pairing Verification Results ===")
print(f"Total JPEG Images:          {len(img_files):,}")
print(f"Total XML Annotations:      {len(xml_files):,}")
print(f"Matched Image-XML Pairs:    {len(common_stems):,}")
print(f"Unmatched Images (no XML):  {len(unmatched_imgs)}")
print(f"Unmatched XMLs (no Image):  {len(unmatched_xmls)}")

# Readability check on a sample of 200 images
print("\nTesting image readability on sample...")
corrupt_samples = []
for stem in common_stems[:200]:
    p = os.path.join(RAW_IMG_DIR, stem + '.jpg')
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        corrupt_samples.append((stem, str(e)))

assert len(corrupt_samples) == 0, f"Corrupt images detected: {corrupt_samples}"
print(f"[PASSED] Sample images verified intact (0 corrupted).")

## Step 8 & 9: Pascal VOC to YOLO Conversion, Coordinate Validation & Class Filtering

Parse each XML file, filter for the 4 target classes (`D00`, `D10`, `D20`, `D40`), validate non-zero bounding box geometries, clamp coordinates, and normalize to YOLO format.

In [ ]:
# 8 & 9. Parse XMLs, filter classes, and normalize bounding boxes
import xml.etree.ElementTree as ET
from collections import defaultdict

# Authoritative 0-indexed YOLO Class Mapping
TARGET_CLASS_MAP = {
    'D00': 0,  # Longitudinal crack
    'D10': 1,  # Transverse crack
    'D20': 2,  # Alligator crack
    'D40': 3   # Pothole
}

CLASS_NAMES = {
    0: 'D00_longitudinal_crack',
    1: 'D10_transverse_crack',
    2: 'D20_alligator_crack',
    3: 'D40_pothole'
}

parsed_data = {}
class_instance_counts = defaultdict(int)
class_image_counts = defaultdict(int)
non_target_filtered = defaultdict(int)
degenerate_boxes_skipped = 0

def convert_voc_xml(xml_path):
    global degenerate_boxes_skipped
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    size_tag = root.find('size')
    w_img = float(size_tag.find('width').text)
    h_img = float(size_tag.find('height').text)
    assert w_img > 0 and h_img > 0, f"Invalid image dimensions: {w_img}x{h_img}"
    
    yolo_boxes = []
    seen_classes = set()
    
    for obj in root.findall('object'):
        name_tag = obj.find('name')
        if name_tag is None or not name_tag.text:
            continue
        label = name_tag.text.strip()
        
        if label in TARGET_CLASS_MAP:
            cid = TARGET_CLASS_MAP[label]
            bnd = obj.find('bndbox')
            xmin = float(bnd.find('xmin').text)
            ymin = float(bnd.find('ymin').text)
            xmax = float(bnd.find('xmax').text)
            ymax = float(bnd.find('ymax').text)
            
            # Clamp coordinates to [0, W] and [0, H]
            xmin_c = max(0.0, min(w_img, xmin))
            ymin_c = max(0.0, min(h_img, ymin))
            xmax_c = max(0.0, min(w_img, xmax))
            ymax_c = max(0.0, min(h_img, ymax))
            
            bw = xmax_c - xmin_c
            bh = ymax_c - ymin_c
            
            # Discard degenerate zero-dimension boxes
            if bw > 1.0 and bh > 1.0:
                xc = ((xmin_c + xmax_c) / 2.0) / w_img
                yc = ((ymin_c + ymax_c) / 2.0) / h_img
                norm_w = bw / w_img
                norm_h = bh / h_img
                
                assert 0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0
                assert 0.0 < norm_w <= 1.0 and 0.0 < norm_h <= 1.0
                
                yolo_boxes.append((cid, xc, yc, norm_w, norm_h))
                class_instance_counts[cid] += 1
                seen_classes.add(cid)
            else:
                degenerate_boxes_skipped += 1
        else:
            non_target_filtered[label] += 1
            
    for cid in seen_classes:
        class_image_counts[cid] += 1
        
    return yolo_boxes

print("Converting XML annotations to normalized YOLO format...")
for stem in tqdm(common_stems, desc="Processing annotations"):
    xml_p = os.path.join(RAW_XML_DIR, stem + '.xml')
    parsed_data[stem] = convert_voc_xml(xml_p)

damage_imgs = sum(1 for boxes in parsed_data.values() if len(boxes) > 0)
sound_imgs = len(parsed_data) - damage_imgs

print("\n=== Conversion & Filtering Metrics ===")
print(f"Total Images Processed:       {len(parsed_data):,}")
print(f"Images with Target Defect(s): {damage_imgs:,}")
print(f"Images without Target Defects: {sound_imgs:,}")
print(f"Degenerate Boxes Discarded:   {degenerate_boxes_skipped}")

print("\nTarget Distress Instances Captured:")
for cid in range(4):
    print(f"  Class {cid} [{CLASS_NAMES[cid]}]: {class_instance_counts[cid]:,} objects across {class_image_counts[cid]:,} images")

print("\nNon-Target Objects Excluded from MVP:")
for cname, count in non_target_filtered.items():
    print(f"  Class {cname}: {count:,} objects excluded")

## Step 10: Sequence-Aware Block Splitting (Preventing Video Leakage)

Dashcam frames were acquired sequentially at 1 frame per second. Adjacent frames share identical backgrounds.  
We group consecutive images into **blocks of 50 frames** and allocate whole blocks to **Train (70%)**, **Validation (15%)**, and **Held-Out Test (15%)** using a fixed random seed (`seed=42`).

In [ ]:
# 10. Sequence-Aware Block Splitting (70% Train, 15% Val, 15% Test)
import random

BLOCK_SIZE = 50
blocks = []
current_block = []

for idx, stem in enumerate(common_stems):
    current_block.append(stem)
    if len(current_block) >= BLOCK_SIZE or idx == len(common_stems) - 1:
        blocks.append(current_block)
        current_block = []

print(f"Created {len(blocks)} continuous sequence blocks (size ~{BLOCK_SIZE} frames).")

# Shuffle blocks deterministically
random.seed(42)
random.shuffle(blocks)

total_blocks = len(blocks)
n_train = int(0.70 * total_blocks)
n_val = int(0.15 * total_blocks)

train_blocks = blocks[:n_train]
val_blocks = blocks[n_train:n_train + n_val]
test_blocks = blocks[n_train + n_val:]

split_assignments = {}
for b in train_blocks: 
    for s in b: split_assignments[s] = 'train'
for b in val_blocks: 
    for s in b: split_assignments[s] = 'val'
for b in test_blocks: 
    for s in b: split_assignments[s] = 'test'

train_stems = [s for s, sp in split_assignments.items() if sp == 'train']
val_stems = [s for s, sp in split_assignments.items() if sp == 'val']
test_stems = [s for s, sp in split_assignments.items() if sp == 'test']

print("\n=== Sequence Split Allocation ===")
print(f"Train Split: {len(train_stems):,} images across {len(train_blocks)} blocks ({len(train_stems)/len(common_stems)*100:.1f}%)")
print(f"Val Split:   {len(val_stems):,} images across {len(val_blocks)} blocks ({len(val_stems)/len(common_stems)*100:.1f}%)")
print(f"Test Split:  {len(test_stems):,} images across {len(test_blocks)} blocks ({len(test_stems)/len(common_stems)*100:.1f}%) [ISOLATED]")

# Assert no cross-contamination
assert len(set(train_stems) & set(val_stems)) == 0, "Data leakage: Train and Val overlap!"
assert len(set(train_stems) & set(test_stems)) == 0, "Data leakage: Train and Test overlap!"
assert len(set(val_stems) & set(test_stems)) == 0, "Data leakage: Val and Test overlap!"
print("[PASSED] Zero frame overlap confirmed across splits.")

## Step 11: YOLO Directory Assembly & `data.yaml` Generation

Populate the standardized YOLO folder structure (`images/{train,val,test}` and `labels/{train,val,test}`) and write the dataset configuration `data.yaml`.

In [ ]:
# 11. Create YOLO directories and save labels and data.yaml
import shutil

YOLO_DATASET_DIR = '/content/visionguard_dataset'

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', split), exist_ok=True)

print("Copying images and writing YOLO label files...")
for stem, split in tqdm(split_assignments.items(), desc="Writing YOLO dataset"):
    # Copy image
    src_img = os.path.join(RAW_IMG_DIR, stem + '.jpg')
    dst_img = os.path.join(YOLO_DATASET_DIR, 'images', split, stem + '.jpg')
    shutil.copyfile(src_img, dst_img)
    
    # Write label text file
    dst_lbl = os.path.join(YOLO_DATASET_DIR, 'labels', split, stem + '.txt')
    boxes = parsed_data[stem]
    with open(dst_lbl, 'w') as f:
        for cid, xc, yc, bw, bh in boxes:
            f.write(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

# Create data.yaml
yaml_content = f"""# VisionGuard AI — RDD2022 YOLOv8 Dataset Configuration
path: {YOLO_DATASET_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: D00_longitudinal_crack
  1: D10_transverse_crack
  2: D20_alligator_crack
  3: D40_pothole
"""

yaml_path = os.path.join(YOLO_DATASET_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n[PASSED] data.yaml created successfully at: {yaml_path}")

## Step 12: Ground-Truth Label Visualization Sanity Check

Read the generated YOLO `.txt` labels back from disk, denormalize bounding boxes to pixel coordinates, and render bounding overlays to visually confirm alignment with genuine road distresses.

In [ ]:
# 12. Visual Sanity Check Grid
import cv2
import matplotlib.pyplot as plt

CLASS_COLORS = {
    0: (240, 50, 50),    # Red: Longitudinal crack
    1: (50, 140, 255),   # Blue: Transverse crack
    2: (40, 210, 40),    # Green: Alligator crack
    3: (255, 175, 0)     # Orange/Amber: Pothole
}

def render_yolo_labels(image_path, label_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cid = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:])
                    xmin = int((xc - bw / 2.0) * w)
                    ymin = int((yc - bh / 2.0) * h)
                    xmax = int((xc + bw / 2.0) * w)
                    ymax = int((yc + bh / 2.0) * h)
                    
                    color = CLASS_COLORS.get(cid, (255, 255, 255))
                    cv2.rectangle(img_rgb, (xmin, ymin), (xmax, ymax), color, 2)
                    lbl_txt = CLASS_NAMES[cid].split('_')[0]
                    cv2.putText(img_rgb, lbl_txt, (xmin, max(18, ymin - 4)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img_rgb

# Select 8 defect-bearing images from train split
train_damage_stems = [s for s in train_stems if len(parsed_data[s]) > 0][:8]

plt.figure(figsize=(16, 8))
for i, stem in enumerate(train_damage_stems):
    img_p = os.path.join(YOLO_DATASET_DIR, 'images', 'train', stem + '.jpg')
    lbl_p = os.path.join(YOLO_DATASET_DIR, 'labels', 'train', stem + '.txt')
    vis = render_yolo_labels(img_p, lbl_p)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(vis)
    plt.title(f"{stem}", fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Step 13: Final Dataset Validation and Audit Report

Compute exact instance tallies per class across splits, verify data integrity assertions, and print the authoritative Phase 3.3 validation report.

In [ ]:
# 13. Complete validation metrics and final assertions
split_counts = defaultdict(lambda: defaultdict(int))
split_bg_imgs = defaultdict(int)

for stem, split in split_assignments.items():
    boxes = parsed_data[stem]
    if len(boxes) == 0:
        split_bg_imgs[split] += 1
    else:
        for cid, _, _, _, _ in boxes:
            split_counts[split][cid] += 1

print("=" * 95)
print("           VISIONGUARD AI — RDD2022 DATASET PREPARATION VALIDATION REPORT")
print("=" * 95)
print(f"{'Split':<8} | {'Images':<8} | {'D00 (Long)':<12} | {'D10 (Trans)':<12} | {'D20 (Allig)':<12} | {'D40 (Pothole)':<14} | {'Background':<10}")
print("-" * 95)

for split in ['train', 'val', 'test']:
    total_s_imgs = len([s for s, sp in split_assignments.items() if sp == split])
    c0 = split_counts[split][0]
    c1 = split_counts[split][1]
    c2 = split_counts[split][2]
    c3 = split_counts[split][3]
    bg = split_bg_imgs[split]
    print(f"{split:<8} | {total_s_imgs:<8} | {c0:<12} | {c1:<12} | {c2:<12} | {c3:<14} | {bg:<10}")

print("=" * 95)

# Final Safety Assertions
assert os.path.exists(yaml_path), "Missing data.yaml file!"
assert len(train_stems) > 0 and len(val_stems) > 0 and len(test_stems) > 0, "One or more splits are empty!"
assert split_counts['train'][0] > 0, "Train split missing class D00!"
assert split_counts['train'][1] > 0, "Train split missing class D10!"
assert split_counts['train'][2] > 0, "Train split missing class D20!"
assert split_counts['train'][3] > 0, "Train split missing class D40!"

print("\n[PASSED] All Phase 3.3 dataset verification criteria satisfied.")
print("The dataset is fully prepared and validated for Phase 3.4 YOLOv8 Nano model training.")

## Step 14: Dataset Packaging & Optional Google Drive Backup

Compress the prepared YOLO dataset (`~1.5 GB`) and optionally copy it to your personal Google Drive so you never need to re-download the 13 GB source archive for future experiments.

In [ ]:
# 14. Optional: Archive prepared dataset and backup to Google Drive
from google.colab import drive
import shutil

ZIP_EXPORT = '/content/visionguard_rdd2022_japan_yolo.zip'
print("Compressing prepared YOLO dataset into zip archive...")
!zip -q -r {ZIP_EXPORT} /content/visionguard_dataset

zip_size_gb = round(os.path.getsize(ZIP_EXPORT) / (1024**3), 2)
print(f"Compressed archive created: {ZIP_EXPORT} ({zip_size_gb} GB)")

# Mount Drive (optional)
try:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/VisionGuard_Artifacts'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    target_backup = os.path.join(DRIVE_DIR, 'visionguard_rdd2022_japan_yolo.zip')
    shutil.copyfile(ZIP_EXPORT, target_backup)
    print(f"[SUCCESS] Dataset successfully saved to persistent Google Drive: {target_backup}")
except Exception as e:
    print(f"Google Drive backup skipped or not mounted: {e}")